In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

from src import processing
from src import analyse_descriptive
from src import regression
from src import visualisation
from src import kmeans

import collecte_donnees

In [ ]:
import importlib
importlib.reload(visualisation)

# Analyse descriptive

## Comparaison des Données de financement des hopitaux des différentes régions de France

In [ ]:
etats_financiers_regionaux = collecte_donnees.generate_etats_financiers_regionaux()
etats_financiers_regionaux.head(5)

In [ ]:
etats_financiers_regionaux = collecte_donnees.generate_finances_regionales()
visualisation.plot_etats_financiers(etats_financiers_regionaux)

In [ ]:
# print(etats_financiers_regionaux["Catégorie"].unique())


## Comparaison des PIB par habitant des différentes régions de France

In [ ]:
# Récupération et traitement des données PIB
pib_brut = collecte_donnees.extract_pib_brut()
pib = processing.prepare_pib_data(pib_brut)
pib_2023 = processing.extract_pib_2023(pib)
print(pib_2023.head())

In [ ]:
# Sélectionner les données de 2023
pib_2023 = processing.extract_pib_2023(pib)
display(pib_2023)

In [ ]:
classement_2023 = analyse_descriptive.classement_regions(pib_2023, "PIB_par_habitant")
visualisation.plot_pib_regions(classement_2023)

In [ ]:
pib_carte = processing.prepare_pib_map(pib_2023)
visualisation.plot_carte_pib(pib_carte)

# Régression

In [ ]:
# Base de données des états financiers (année 2023)
etats_financiers = collecte_donnees.generate_finances_regionales()
df_financier = etats_financiers.copy()
display(df_financier.head())

In [ ]:
print(df_financier)

A ce stade, nos deux bases de données pib_2023 (PIB par habitant en 2023) et df_financier ( états financier des régions en 2023) contiennent le même nombre derégions avec les mêmes noms. On souhaite effectuer des regressions afin d'observer la correlation entre le niveau de richesse des régions et les indicatuers des dépenses. 
On effectue donc des regressions simples unidimentionnels des indicateurs de dépenses les plus pertnents, à savoir les dÉpenses des régimes d'assurance maladoe et le FIR (Fond d'intervention régional)

$$ \widehat{Y}_i = \widehat{\beta}_0 + \widehat{\beta}_1 X_i $$

Avec :
- $\widehat{Y}_i$ : les dépenses de santé estimées pour la région $i$ ;
- $X_i$ : le PIB par habitant de la région $i$ ;
- $\widehat{\beta}_0$ : la constante estimée ;
- $\widehat{\beta}_1$ : le coefficient directeur estimé, qui mesure l'association entre le PIB par habitant et les dépenses de santé.



In [ ]:
print(df_financier.columns.tolist())

In [ ]:
# Ajouter le PIB par habitant
df_analyse = pd.merge( df_financier, pib_2023, on="Région", how="inner" )

In [ ]:
print("Colonnes df_financier :", df_financier.columns.tolist())
print("Colonnes pib_2023 :", pib_2023.columns.tolist())
print("Colonnes df_analyse :", df_analyse.columns.tolist())

In [ ]:
# Variable explicative commune aux trois régressions
X = df_analyse[["PIB_par_habitant"]]
X = sm.add_constant(X)

variable_explicative = "PIB_par_habitant"
variables_expliquees = [
    "DEPENSES DES REGIMES D'ASSURANCE MALADIE (hors contribution aux dépenses des ESMS pour personnes âgées et handicapées)",
    "DEPENSES DES REGIMES D'ASSURANCE MALADIE ET DE LA CAISSE NATIONALE DE SOLIDARITE POUR L'AUTONOMIE (décaissements)",
    "FONDS D'INTERVENTION REGIONAL (FIR)"
]

In [ ]:
# Réalisation des régressions

resultats = regression.plusieurs_regressions(df_analyse, variable_explicative, variables_expliquees)

for variable, modele in resultats.items():
    print("\n" + "=" * 80)
    print("VARIABLE EXPLIQUÉE :", variable)
    print("=" * 80)
    print(modele.summary())

In [ ]:
# Tracé des trois régressions

for variable, modele in resultats.items():
    visualisation.plot_regression(df_analyse, modele, variable_explicative, variable,f"regression de : {variable}")

# 4. Identification des profils des établissements de snaté

Pour cette partie, nous utilisons les données de la SAE (Statistique Annuelle des Établissements de santé) de 2023. Cette base regroupe des informations sur les établissements de santé français, notamment leur activité, leurs capacités d'accueil et leur personnel. Nous nous intéressons plus particulièrement à la table MCO (Médecine, Chirurgie et Obstétrique), qui contient des données sur les lits disponibles, les séjours hospitaliers et les journées d'hospitalisation, etc.

À l'aide du fichier Excel qui accompagne la base, nous pouvons associer chaque nom de variable à sa signification. Cette étape nous permet de comprendre les informations disponibles et de sélectionner les variables les plus pertinentes pour notre étude. L'objectif est de retenir des indicateurs qui permettront de caractériser les établissements de santé et de les regrouper selon leurs profils grâce à une méthode de classification (K-means).

In [ ]:
# On récupère le dictionnaire de variables
mco = projet.extract_mco_sae()
dictionnaire = projet.extract_dictionnaire_sae()
dic = dictionnaire["SAE 2023"]

# On filtre uniquement les variables de la table MCO
dic_mco = dic[dic["Table SAS"] == "MCO"]

# On récupère les colonnes présentes dans notre CSV
colonnes_mco = pd.DataFrame({"Variable SAS": mco.columns})

# On fait la correspondance avec le dictionnaire Excel
signification_mco = colonnes_mco.merge(dic_mco[["Variable SAS", "Label variable SAS"]], how="left")

In [ ]:
# Signification des colonnes de la base de données
pd.set_option("display.max_colwidth", None)
display(signification_mco)

In [ ]:
mco_selection, mco_standardise = processing.prepare_mco_data(mco)
#Vérification des doublons
print("Nombre de doublons :", mco.duplicated(subset="FI").sum())

In [ ]:
#Application de la méthode kmeans :
kmeans.methode_coude(mco_standardise)

On choisit alors 3 comme nombre de clusters, par la méthode du coude.

In [ ]:
clusters = kmeans.appliquer_kmeans(mco_standardise, 3)
mco_selection["Cluster"] = clusters

In [ ]:
print(mco_selection["Cluster"].value_counts().sort_index())

In [ ]:
profil_clusters = mco_selection.groupby("Cluster")[
    ["LIT_MCO", "JLI_MCO", "SEJHC_MCO", "JOU_MCO",
     "PLA_MCO", "SEJHP_MCO", "SEA_MCO"]
].mean()

print(profil_clusters)

On observe que le cluster 0 correspond à peu de lits et une activité relativement faible, que le cluster 1 correspond à beaucoup de lits, de journées et de séjours, il regroupe les établissements ayant les volumes d'activité les plus importants. Le Cluster 2 regroupe quand à lui des valeurs intermédiaires entre les deux autres groupes.

### Répartition par département

Le numéro FINESS (Fichier National des Établissements Sanitaires et Sociaux) est un identifiant unique de 9 caractères dont disposent les établissements sanitaire. En général, ses deux premiers caractères indiquent le département dans lequel il se situe. C'est ce qui va nous permettre de le localiser sur la carte des départements de France.

In [ ]:
# On récupère le FINESS et le departement
mco_selection["FI"] = mco.loc[mco_selection.index, "FI"].astype(str).str.zfill(9)
mco_selection["Departement"] = mco_selection["FI"].apply(
    lambda x: x[:3] if x.startswith("97") else x[:2]
)

#On vérifie les départements
print(sorted(mco_selection["Departement"].unique()))

In [ ]:
#Répartition de sclusters par département
repartition_departements = mco_selection.groupby(
    ["Departement", "Cluster"]
).size().unstack(fill_value=0)

repartition_departements.head()

In [ ]:
visualisation.visualiser_clusters_departements(mco_selection)

On peut, grace à cette répartition, visualiser le pourcentage de chaque cluster sur la carte des départements.

In [ ]:
#Visualisation sur la carte
visualisation.visualiser_clusters_carte( mco_selection )

### Visualisation du nombre de lit par personne et par region

mco = Medecine Chirurgie Obstetrique

lit_med = lits de medecine installes en hospitalisation completes au 31/12


In [ ]:
mco = collecte_donnees.epure_mco()
visualisation.plot_mco(mco)

### Visualisation du taux de pauvreté par région

In [ ]:
pauvrete = collecte_donnees.pauvrete()
visualisation.plot_pauvrete(pauvrete)